In [ ]:
"""
m/z-to-m/z Spatial Pattern Matching Pipeline for Isotope Detection
===================================================================
Compares each m/z feature with all other m/z features within the same sample
to identify isotope patterns based on spatial similarity scores.

Based on Gene-to-m/z Spatial Pattern Matching Pipeline V1 (Analytic - Optimized)
Modified to:
- Compare m/z features within each sample (no cross-modal alignment)
- Remove 180-degree rotation (same coordinate space)
- Skip visualizations
- Output similarity scores to CSV
"""

import numpy as np
import scanpy as sc
from sklearn.neighbors import NearestNeighbors
from scipy.stats import pearsonr
from typing import Dict, Optional
import pandas as pd
import os
import warnings
from dataclasses import dataclass
from joblib import Parallel, delayed
from tqdm import tqdm

warnings.filterwarnings('ignore')

# =============================================================================
# DATA CONFIGURATION
# =============================================================================

MSI_PIXEL_SIZE = 60  # μm

# Mass differences for isotope and adduct detection
MASS_DIFFS = {
    'Isotope (M+1)': 1.0033,
    'Isotope (M+2)': 2.0067,
    'Adduct (NH4)': 17.0265,
    'Adduct (Na)': 21.982,
    'Adduct (K)': 37.9555
}

# Tolerance for mass difference matching (in Da)
MASS_DIFF_TOLERANCE = 0.01

# Minimum score percentage for high-confidence pairs (relative to self-score)
HIGH_CONFIDENCE_PERCENTAGE = 80  # %

MSI_INPUT_FOLDER = "/home/ajarrah/PhD_Thesis/chapter_2/h5ad_data_processed_4lockmasses_filtered_halfbrain_common/"
MSI_SAMPLE_FILES = [
    "halfbrain_yc_1_filtered_common.h5ad", "halfbrain_yc_2_filtered_common.h5ad",
    "halfbrain_yc_3_filtered_common.h5ad", "halfbrain_yc_4_filtered_common.h5ad",
    "halfbrain_yad_1_filtered_common.h5ad", "halfbrain_yad_2_filtered_common.h5ad",
    "halfbrain_yad_3_filtered_common.h5ad", "halfbrain_yad_4_filtered_common.h5ad",
    "halfbrain_ac_1_filtered_common.h5ad", "halfbrain_ac_2_filtered_common.h5ad",
    "halfbrain_ac_3_filtered_common.h5ad", "halfbrain_ac_4_filtered_common.h5ad",
    "halfbrain_aad_1_filtered_common.h5ad", "halfbrain_aad_2_filtered_common.h5ad",
    "halfbrain_aad_3_filtered_common.h5ad", "halfbrain_aad_4_filtered_common.h5ad"
]
MSI_SAMPLE_IDS = [
    "YC_1", "YC_2", "YC_3", "YC_4",
    "YAD_1", "YAD_2", "YAD_3", "YAD_4",
    "AC_1", "AC_2", "AC_3", "AC_4",
    "AAD_1", "AAD_2", "AAD_3", "AAD_4"
]


def compute_spatial_histogram(coords: np.ndarray, values: np.ndarray, n_bins: int = 10) -> np.ndarray:
    coord_min, coord_max = coords.min(axis=0), coords.max(axis=0)
    norm = (coords - coord_min) / (coord_max - coord_min + 1e-8)
    x_bins = np.clip((norm[:, 0] * n_bins).astype(int), 0, n_bins - 1)
    y_bins = np.clip((norm[:, 1] * n_bins).astype(int), 0, n_bins - 1)
    flat_idx = y_bins * n_bins + x_bins
    hist = np.bincount(flat_idx, weights=values, minlength=n_bins * n_bins).reshape(n_bins, n_bins)
    counts = np.bincount(flat_idx, minlength=n_bins * n_bins).reshape(n_bins, n_bins)
    hist = np.divide(hist, counts, where=counts > 0, out=np.zeros_like(hist))
    hist_min, hist_max = hist.min(), hist.max()
    if hist_max > hist_min:
        hist = (hist - hist_min) / (hist_max - hist_min)
    return hist


def compute_radial_profile(coords: np.ndarray, values: np.ndarray, n_rings: int = 10) -> np.ndarray:
    coord_min, coord_max = coords.min(axis=0), coords.max(axis=0)
    norm = (coords - coord_min) / (coord_max - coord_min + 1e-8)
    centroid = norm.mean(axis=0)
    distances = np.linalg.norm(norm - centroid, axis=1)
    max_dist = distances.max() + 1e-8
    ring_idx = np.clip((distances / max_dist * n_rings).astype(int), 0, n_rings - 1)
    profile = np.bincount(ring_idx, weights=values, minlength=n_rings)
    counts = np.bincount(ring_idx, minlength=n_rings)
    profile = np.divide(profile, counts, where=counts > 0, out=np.zeros_like(profile, dtype=float))
    prof_min, prof_max = profile.min(), profile.max()
    if prof_max > prof_min:
        profile = (profile - prof_min) / (prof_max - prof_min)
    return profile


def compute_morans_i_vectorized(coords: np.ndarray, values: np.ndarray, indices: np.ndarray) -> float:
    n = len(values)
    mean_val = values.mean()
    deviations = values - mean_val
    denom = np.sum(deviations ** 2)
    if denom == 0:
        return 0.0
    neighbor_deviations = deviations[indices[:, 1:]]
    numer = np.sum(deviations[:, np.newaxis] * neighbor_deviations)
    w_sum = indices.shape[0] * (indices.shape[1] - 1)
    return (n / w_sum) * (numer / denom) if w_sum > 0 else 0.0


@dataclass
class SpatialSignature:
    sample_id: str
    feature_name: str
    feature_type: str
    node_importance: np.ndarray
    spatial_histogram: np.ndarray = None
    radial_profile: np.ndarray = None
    morans_i: float = 0.0
    coordinates: np.ndarray = None
    raw_values: np.ndarray = None


def compute_coordinate_based_similarity(sig1: SpatialSignature, sig2: SpatialSignature, grid_res: int = 50) -> dict:
    """Compute coordinate-based similarity between two signatures in the same coordinate space."""
    coords = sig1.coordinates  # Same coordinates for both (same sample)
    
    # Direct value correlation (no grid interpolation needed - same coordinates)
    mask = (sig1.raw_values > 0) | (sig2.raw_values > 0)
    value_corr = 0
    if mask.sum() > 10:
        r, _ = pearsonr(sig1.raw_values, sig2.raw_values)
        value_corr = r if not np.isnan(r) else 0
    
    # Importance correlation
    imp_corr = 0
    if len(sig1.node_importance) > 10:
        r, _ = pearsonr(sig1.node_importance, sig2.node_importance)
        imp_corr = r if not np.isnan(r) else 0
    
    # Importance IoU
    imp1 = sig1.node_importance / (sig1.node_importance.max() + 1e-8)
    imp2 = sig2.node_importance / (sig2.node_importance.max() + 1e-8)
    importance_iou = np.minimum(imp1, imp2).sum() / (np.maximum(imp1, imp2).sum() + 1e-8)
    
    # Intensity ratio consistency (key for isotope detection)
    # For true isotopes, the ratio between intensities should be consistent across pixels
    intensity_ratio_consistency = compute_intensity_ratio_consistency(sig1.raw_values, sig2.raw_values)
    
    # Peak colocalization - percentage of high-intensity pixels that overlap
    peak_colocalization = compute_peak_colocalization(sig1.raw_values, sig2.raw_values)
    
    return {
        'value_correlation': value_corr, 
        'importance_iou': importance_iou, 
        'importance_correlation': imp_corr,
        'intensity_ratio_consistency': intensity_ratio_consistency,
        'peak_colocalization': peak_colocalization
    }


def compute_intensity_ratio_consistency(values1: np.ndarray, values2: np.ndarray, min_intensity_pct: float = 10) -> float:
    """
    Compute how consistent the intensity ratio is across pixels.
    For true isotopes, M+1/M+0 ratio should be relatively constant.
    Returns a score from 0 to 1, where 1 = perfectly consistent ratio.
    """
    # Only consider pixels where both have signal above threshold
    thresh1 = np.percentile(values1, min_intensity_pct)
    thresh2 = np.percentile(values2, min_intensity_pct)
    
    mask = (values1 > thresh1) & (values2 > thresh2)
    
    if mask.sum() < 10:
        return 0.0
    
    v1 = values1[mask]
    v2 = values2[mask]
    
    # Compute ratio (use smaller/larger to keep ratio <= 1)
    ratios = np.minimum(v1, v2) / (np.maximum(v1, v2) + 1e-8)
    
    # Consistency = 1 - coefficient of variation of the ratio
    # Lower CV = more consistent ratio = higher score
    ratio_mean = ratios.mean()
    ratio_std = ratios.std()
    
    if ratio_mean > 0:
        cv = ratio_std / ratio_mean
        # Convert CV to 0-1 score (CV of 0 = score of 1, CV of 1+ = score ~0)
        consistency = 1 / (1 + cv)
    else:
        consistency = 0.0
    
    return consistency


def compute_peak_colocalization(values1: np.ndarray, values2: np.ndarray, top_pct: float = 20) -> float:
    """
    Compute overlap of high-intensity regions between two m/z features.
    Returns IoU of top percentile pixels.
    """
    thresh1 = np.percentile(values1, 100 - top_pct)
    thresh2 = np.percentile(values2, 100 - top_pct)
    
    peaks1 = values1 >= thresh1
    peaks2 = values2 >= thresh2
    
    intersection = (peaks1 & peaks2).sum()
    union = (peaks1 | peaks2).sum()
    
    if union > 0:
        return intersection / union
    return 0.0


def compute_descriptor_similarity(sig1: SpatialSignature, sig2: SpatialSignature) -> dict:
    def safe_pearsonr(a, b):
        if a.std() > 0 and b.std() > 0:
            r, _ = pearsonr(a, b)
            return r if not np.isnan(r) else 0
        return 0
    
    spatial_hist_corr = safe_pearsonr(sig1.spatial_histogram.flatten(), sig2.spatial_histogram.flatten())
    radial_corr = safe_pearsonr(sig1.radial_profile, sig2.radial_profile)
    morans_sim = 1 / (1 + abs(sig1.morans_i - sig2.morans_i))
    
    return {'spatial_hist_corr': spatial_hist_corr,
            'radial_corr': radial_corr, 
            'morans_similarity': morans_sim}


def compute_combined_score(coord_sim: dict, desc_sim: dict) -> float:
    """
    Compute combined score optimized for isotope detection.
    
    Weights adjusted to prioritize:
    - intensity_ratio_consistency (highest - key isotope indicator)
    - value_correlation (high - isotopes correlate spatially)
    - peak_colocalization (high - isotopes share hotspots)
    - spatial_hist_corr, radial_corr (medium - spatial pattern similarity)
    - importance metrics (lower - supplementary)
    - morans_similarity (lowest - general spatial autocorrelation)
    """
    # Coordinate-based metrics (60% total)
    coord_score = (
        0.25 * max(coord_sim['intensity_ratio_consistency'], 0) +  # Key isotope metric
        0.15 * max(coord_sim['value_correlation'], 0) +
        0.10 * max(coord_sim['peak_colocalization'], 0) +
        0.05 * coord_sim['importance_iou'] +
        0.05 * max(coord_sim['importance_correlation'], 0)
    )
    
    # Descriptor-based metrics (40% total)
    desc_score = (
        0.18 * max(desc_sim['spatial_hist_corr'], 0) +
        0.18 * max(desc_sim['radial_corr'], 0) +
        0.04 * desc_sim['morans_similarity']
    )
    
    return coord_score + desc_score


def classify_mass_difference(mz_diff: float, tolerance: float = MASS_DIFF_TOLERANCE) -> str:
    """Classify the mass difference into isotope/adduct type."""
    for label, expected_diff in MASS_DIFFS.items():
        if abs(mz_diff - expected_diff) <= tolerance:
            return label
    return 'None'


class MzIsotopeMatcher:
    def __init__(self, output_dir: str = './mz_isotope_matching_results', n_jobs: int = -1):
        self.output_dir = output_dir
        self.n_jobs = n_jobs
        os.makedirs(output_dir, exist_ok=True)
        self.msi_data = {}
        self._nn_cache = {}

    def load_all_data(self):
        print(f"Loading MSI data...")
        print(f"  Pixel size: {MSI_PIXEL_SIZE} μm (Cartesian)")
        for file, sample_id in zip(MSI_SAMPLE_FILES, MSI_SAMPLE_IDS):
            path = os.path.join(MSI_INPUT_FOLDER, file)
            if os.path.exists(path):
                self.msi_data[sample_id] = sc.read_h5ad(path)
                print(f"  {sample_id}: {self.msi_data[sample_id].shape}")
            else:
                print(f"  {sample_id}: NOT FOUND at {path}")

    def _get_nn_indices(self, coords: np.ndarray, k: int, cache_key: str) -> np.ndarray:
        full_key = f"{cache_key}_{k}"
        if full_key not in self._nn_cache:
            coord_min, coord_max = coords.min(axis=0), coords.max(axis=0)
            norm = (coords - coord_min) / (coord_max - coord_min + 1e-8)
            k_actual = min(k + 1, len(coords))
            nn = NearestNeighbors(n_neighbors=k_actual)
            nn.fit(norm)
            _, indices = nn.kneighbors(norm)
            self._nn_cache[full_key] = indices
        return self._nn_cache[full_key]

    def compute_bio_importance(self, coords: np.ndarray, values: np.ndarray, k: int, nn_indices: np.ndarray) -> np.ndarray:
        neighbor_vals = values[nn_indices[:, 1:]]
        local_var = np.var(neighbor_vals, axis=1)
        lv_min, lv_max = local_var.min(), local_var.max()
        if lv_max > lv_min:
            local_var = (local_var - lv_min) / (lv_max - lv_min)
        else:
            local_var = np.zeros_like(local_var)
        v_min, v_max = values.min(), values.max()
        if v_max > v_min:
            val_norm = (values - v_min) / (v_max - v_min)
        else:
            val_norm = np.zeros_like(values)
        return 0.5 * local_var + 0.5 * val_norm

    def extract_signature(self, coords: np.ndarray, values: np.ndarray, sample_id: str,
                          feature_name: str, n_neighbors: int, nn_indices: np.ndarray) -> SpatialSignature:
        bio_imp = self.compute_bio_importance(coords, values, n_neighbors, nn_indices)
        return SpatialSignature(
            sample_id=sample_id, feature_name=feature_name, feature_type='mz',
            node_importance=bio_imp,
            spatial_histogram=compute_spatial_histogram(coords, values),
            radial_profile=compute_radial_profile(coords, values),
            morans_i=compute_morans_i_vectorized(coords, values, nn_indices),
            coordinates=coords, raw_values=values)

    def compute_pair_similarity(self, sig1: SpatialSignature, sig2: SpatialSignature) -> dict:
        """Compute all similarity metrics between two m/z signatures."""
        coord_sim = compute_coordinate_based_similarity(sig1, sig2)
        desc_sim = compute_descriptor_similarity(sig1, sig2)
        combined = compute_combined_score(coord_sim, desc_sim)
        
        # Calculate m/z difference for isotope detection
        try:
            mz1 = float(sig1.feature_name)
            mz2 = float(sig2.feature_name)
            mz_diff = abs(mz2 - mz1)
            # Classify the mass difference
            mass_diff_type = classify_mass_difference(mz_diff)
        except ValueError:
            mz_diff = np.nan
            mass_diff_type = 'None'
        
        return {
            'mz_1': sig1.feature_name,
            'mz_2': sig2.feature_name,
            'mz_difference': mz_diff,
            'mass_diff_type': mass_diff_type,
            'sample_id': sig1.sample_id,
            **coord_sim,
            **desc_sim,
            'combined_score': combined,
            'mz_1_morans_i': sig1.morans_i,
            'mz_2_morans_i': sig2.morans_i
        }

    def run_analysis(self):
        print("\n" + "=" * 70)
        print("m/z-to-m/z MATCHING FOR ISOTOPE DETECTION")
        print(f"MSI: {MSI_PIXEL_SIZE}μm (Cartesian)")
        print(f"Mass differences: {list(MASS_DIFFS.keys())}")
        print(f"Tolerance: ±{MASS_DIFF_TOLERANCE} Da")
        print(f"High-confidence threshold: {HIGH_CONFIDENCE_PERCENTAGE}% of self-score")
        print("=" * 70)

        all_results = []

        for sample_id in tqdm(MSI_SAMPLE_IDS, desc="Samples", unit="sample"):
            if sample_id not in self.msi_data:
                print(f"\n{sample_id}: NOT LOADED, skipping")
                continue

            print(f"\n{'=' * 50}")
            print(f"Sample: {sample_id}")
            print(f"{'=' * 50}")

            msi_adata = self.msi_data[sample_id]
            msi_coords = np.column_stack([msi_adata.obs['x_um'].values, msi_adata.obs['y_um'].values])
            msi_data = msi_adata.X.toarray() if hasattr(msi_adata.X, 'toarray') else msi_adata.X
            mz_names = list(msi_adata.var_names)
            n_mz = len(mz_names)

            print(f"  {n_mz} m/z features, {len(msi_coords)} pixels")

            # Pre-compute nearest neighbors
            nn_indices = self._get_nn_indices(msi_coords, 8, f"msi_{sample_id}")

            # Compare only pairs with relevant mass differences
            print(f"  Finding candidate isotope/adduct pairs...")
            
            # Parse m/z values
            mz_values = []
            for name in mz_names:
                try:
                    mz_values.append(float(name))
                except ValueError:
                    mz_values.append(np.nan)
            mz_values = np.array(mz_values)
            
            # Find pairs with mass differences matching isotope/adduct patterns
            candidate_pairs = []
            for i in range(n_mz):
                if np.isnan(mz_values[i]):
                    continue
                for j in range(i + 1, n_mz):
                    if np.isnan(mz_values[j]):
                        continue
                    mz_diff = abs(mz_values[j] - mz_values[i])
                    # Check if mass difference matches any isotope/adduct pattern
                    for diff_type, expected_diff in MASS_DIFFS.items():
                        if abs(mz_diff - expected_diff) <= MASS_DIFF_TOLERANCE:
                            candidate_pairs.append((i, j, diff_type))
                            break
            
            n_candidates = len(candidate_pairs)
            print(f"  {n_candidates} candidate pairs (from {n_mz * (n_mz - 1) // 2} total possible)")
            
            if n_candidates == 0:
                print(f"  No candidate pairs found, skipping sample")
                continue
            
            # Extract signatures only for m/z features involved in candidate pairs
            involved_indices = set()
            for i, j, _ in candidate_pairs:
                involved_indices.add(i)
                involved_indices.add(j)
            involved_indices = sorted(involved_indices)
            
            print(f"  Extracting signatures for {len(involved_indices)} involved m/z features...")
            def extract_single_mz(i):
                return i, self.extract_signature(msi_coords, msi_data[:, i], sample_id, mz_names[i], 8, nn_indices)

            sig_results = Parallel(n_jobs=self.n_jobs, prefer='threads')(
                delayed(extract_single_mz)(i) for i in tqdm(involved_indices, desc="  Signatures", unit="mz"))
            
            signatures = {i: sig for i, sig in sig_results}
            print(f"  {len(signatures)} signatures extracted")

            # Compute self-scores for each m/z (maximum possible score)
            print(f"  Computing self-scores...")
            self_scores = {}
            for i in involved_indices:
                self_score_result = self.compute_pair_similarity(signatures[i], signatures[i])
                self_scores[i] = self_score_result['combined_score']
            
            # Compute similarities for candidate pairs
            print(f"  Computing pairwise similarities...")
            
            def compute_pair(i, j, diff_type):
                result = self.compute_pair_similarity(signatures[i], signatures[j])
                result['mass_diff_type'] = diff_type  # Override with pre-determined type
                # Add self-scores and percentage
                result['mz_1_self_score'] = self_scores[i]
                result['mz_2_self_score'] = self_scores[j]
                max_self_score = max(self_scores[i], self_scores[j])
                result['score_percentage'] = (result['combined_score'] / max_self_score * 100) if max_self_score > 0 else 0
                return result

            # Parallel computation with progress bar
            pair_results = Parallel(n_jobs=self.n_jobs, prefer='threads')(
                delayed(compute_pair)(i, j, diff_type) for i, j, diff_type in tqdm(candidate_pairs, desc="  Pairs", unit="pair"))

            # Convert to DataFrame
            sample_results = pd.DataFrame(pair_results)
            all_results.append(sample_results)

            print(f"  Sample complete: {len(sample_results)} pairs scored")

        # Save all results
        if all_results:
            print("\n" + "=" * 70)
            print("SAVING RESULTS")
            print("=" * 70)

            # Full pairwise results (only candidate pairs now)
            full_results = pd.concat(all_results, ignore_index=True)
            full_results = full_results.sort_values(['sample_id', 'mass_diff_type', 'combined_score'], ascending=[True, True, False])
            full_path = os.path.join(self.output_dir, 'mz_to_mz_isotope_candidates.csv')
            full_results.to_csv(full_path, index=False)
            print(f"  All candidate pairs: {full_path} ({len(full_results)} rows)")

            # Summary statistics
            summary = []
            for sample_id in MSI_SAMPLE_IDS:
                if sample_id in self.msi_data:
                    sample_data = full_results[full_results['sample_id'] == sample_id]
                    summary.append({
                        'sample_id': sample_id,
                        'n_mz_features': len(self.msi_data[sample_id].var_names),
                        'n_pairs': len(sample_data),
                        'mean_combined_score': sample_data['combined_score'].mean(),
                        'max_combined_score': sample_data['combined_score'].max(),
                        'min_combined_score': sample_data['combined_score'].min(),
                        'std_combined_score': sample_data['combined_score'].std()
                    })
            
            summary_df = pd.DataFrame(summary)
            summary_path = os.path.join(self.output_dir, 'mz_matching_summary.csv')
            summary_df.to_csv(summary_path, index=False)
            print(f"  Summary: {summary_path}")

            # High-confidence isotope/adduct pairs (score >= percentage threshold of self-score)
            high_conf_candidates = full_results[full_results['score_percentage'] < HIGH_CONFIDENCE_PERCENTAGE].copy()
            high_conf_candidates = high_conf_candidates.sort_values(
                ['mass_diff_type', 'score_percentage'], ascending=[True, False])
            high_conf_path = os.path.join(self.output_dir, 'high_confidence_isotope_adduct_pairs.csv')
            high_conf_candidates.to_csv(high_conf_path, index=False)
            print(f"  High-confidence pairs (>={HIGH_CONFIDENCE_PERCENTAGE}% of self-score): {high_conf_path} ({len(high_conf_candidates)} pairs)")

            # Print summary by type
            if len(high_conf_candidates) > 0:
                print("\n  High-confidence pairs by type:")
                for diff_type in MASS_DIFFS.keys():
                    type_count = len(high_conf_candidates[
                        high_conf_candidates['mass_diff_type'] == diff_type])
                    if type_count > 0:
                        print(f"    {diff_type}: {type_count} pairs")

            return full_results
        
        return None


def main():
    print("=" * 70)
    print("m/z-to-m/z Isotope Pattern Matching")
    print(f"MSI: {MSI_PIXEL_SIZE}μm")
    print("=" * 70)
    
    matcher = MzIsotopeMatcher(output_dir='./2_mz_isotope_not_80_matching_results', n_jobs=-1)
    matcher.load_all_data()
    results = matcher.run_analysis()
    
    print("\n" + "=" * 70)
    print("COMPLETE!")
    print("=" * 70)
    
    return matcher, results


if __name__ == "__main__":
    matcher, results = main()

m/z-to-m/z Isotope Pattern Matching
MSI: 60μm
Loading MSI data...
  Pixel size: 60 μm (Cartesian)
  YC_1: (6688, 528)
  YC_2: (7858, 528)
  YC_3: (7150, 528)
  YC_4: (6067, 528)
  YAD_1: (7517, 528)
  YAD_2: (7596, 528)
  YAD_3: (6844, 528)
  YAD_4: (7591, 528)
  AC_1: (6955, 528)
  AC_2: (5729, 528)
  AC_3: (7569, 528)
  AC_4: (7792, 528)
  AAD_1: (6471, 528)
  AAD_2: (5959, 528)
  AAD_3: (5392, 528)
  AAD_4: (6833, 528)

m/z-to-m/z MATCHING FOR ISOTOPE DETECTION
MSI: 60μm (Cartesian)
Mass differences: ['Isotope (M+1)', 'Isotope (M+2)', 'Adduct (NH4)', 'Adduct (Na)', 'Adduct (K)']
Tolerance: ±0.01 Da
High-confidence threshold: 70% of self-score


Samples:   0%|          | 0/16 [00:00<?, ?sample/s]


Sample: YC_1
  528 m/z features, 6688 pixels
  Finding candidate isotope/adduct pairs...
  786 candidate pairs (from 139128 total possible)
  Extracting signatures for 492 involved m/z features...


  Signatures: 100%|██████████| 492/492 [00:00<00:00, 1093.09mz/s]


  492 signatures extracted
  Computing self-scores...
  Computing pairwise similarities...


...

  Signatures: 100%|██████████| 493/493 [00:01<00:00, 357.15mz/s]


  493 signatures extracted
  Computing self-scores...
  Computing pairwise similarities...


Samples: 100%|██████████| 16/16 [02:53<00:00, 10.87s/sample]

  Sample complete: 785 pairs scored

SAVING RESULTS


  All candidate pairs: ./2__mz_isotope_matching_results/mz_to_mz_isotope_candidates.csv (12527 rows)
  Summary: ./2__mz_isotope_matching_results/mz_matching_summary.csv
  High-confidence pairs (>=70% of self-score): ./2__mz_isotope_matching_results/high_confidence_isotope_adduct_pairs.csv (6373 pairs)

  High-confidence pairs by type:
    Isotope (M+1): 3484 pairs
    Isotope (M+2): 1892 pairs
    Adduct (NH4): 212 pairs
    Adduct (Na): 539 pairs
    Adduct (K): 246 pairs

COMPLETE!
